# Import Statements

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/distilbert-weights/distilbert-1/config.json
/kaggle/input/distilbert-weights/distilbert-1/tokenizer.json
/kaggle/input/distilbert-weights/distilbert-1/tokenizer_config.json
/kaggle/input/distilbert-weights/distilbert-1/model.safetensors
/kaggle/input/distilbert-weights/distilbert-1/special_tokens_map.json
/kaggle/input/distilbert-weights/distilbert-1/vocab.txt
/kaggle/input/processed/train_lemmastop.csv
/kaggle/input/processed/cleaned_train.csv
/kaggle/input/processed/test_lemmastop.csv
/kaggle/input/processed/cleaned_test.csv
/kaggle/input/2025-sep-dl-gen-ai-project/sample_submission.csv
/kaggle/input/2025-sep-dl-gen-ai-project/train.csv
/kaggle/input/2025-sep-dl-gen-ai-project/test.csv
/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/config.json
/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/training_args.bin
/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/tokenizer.json
/kaggle/input/d-bert-train/transformers/default/1/d_ber

In [2]:
# !pip install contractions

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('punkt')

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

import torch
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from tqdm.auto import tqdm

2025-11-28 19:32:21.535382: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764358341.739944      38 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764358341.799108      38 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# Set Device

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
emotion_cols = ['anger', 'fear', 'joy', 'sadness', 'surprise']

# Emotion Dataset Class

In [8]:
class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        labels = torch.tensor(self.labels[idx], dtype=torch.float32)

        # Tokenize the text
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': labels
        }

In [21]:
os.environ["WANDB_DISABLED"] = "true"

# Load Tokenizer and Model

In [12]:
MODEL_DIR = "/kaggle/input/distilbert-weights/distilbert-1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification"
)
model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [22]:
inference_args = TrainingArguments(
    output_dir="/kaggle/working",
    per_device_eval_batch_size=16,
    dataloader_drop_last=False,
    report_to=None
)

trainer = Trainer(
    model=model,
    args=inference_args,
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


# Test Dataset

In [13]:
df_test = pd.read_csv('/kaggle/input/2025-sep-dl-gen-ai-project/test.csv')
test_texts = df_test['text'].tolist()

In [14]:
num_labels = len(emotion_cols)
dummy_labels = [[0] * num_labels for _ in range(len(test_texts))]

# Make Predictions

In [23]:
test_dataset = EmotionDataset(texts=test_texts, labels=dummy_labels, tokenizer=tokenizer)
raw_predictions = trainer.predict(test_dataset)

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


In [24]:
test_logits = raw_predictions.predictions
sigmoid_preds = torch.sigmoid(torch.Tensor(test_logits))
binary_preds = (sigmoid_preds > 0.5).int().numpy()
print(binary_preds[:5])

[[1 1 0 1 0]
 [0 0 0 0 0]
 [0 1 0 0 1]
 [0 1 0 0 0]
 [0 1 0 0 1]]


# Submission

In [25]:
df_submission = pd.DataFrame(binary_preds, columns=emotion_cols)
df_submission[emotion_cols] = df_submission[emotion_cols].astype('int64')
df_submission['id'] = range(len(df_test))
df_submission = df_submission[['id', 'anger', 'fear', 'joy', 'sadness', 'surprise']]
df_submission.head()

,id,anger,fear,joy,sadness,surprise
0,0,1,1,0,1,0
1,1,0,0,0,0,0
2,2,0,1,0,0,1
3,3,0,1,0,0,0
4,4,0,1,0,0,1


In [26]:
df_submission.to_csv('submission.csv', index=False)